# 🧬 LifeOps: GRPO Training — Chaotic Life Management

Trains `Qwen2.5-3B-Instruct` with GRPO to balance Career, Family, and Health.

Run this notebook on **Colab T4 (free tier)**.

**Environment:** In section 3, set `LIFEOPS_SPACE_URL` to your Hugging Face Space root (same pattern as a hosted OpenEnv URL). Leave it empty to start the server locally instead.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade "trl>=0.15.0" transformers datasets httpx matplotlib openenv-core python-dotenv mergekit

## 2. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print("✅ Model loaded")

## 3. Connect LifeOps API (Hugging Face Space or local server)

In [ ]:
import os
import subprocess
import time
import httpx
import sys

# 1. Extract and Navigate
if os.path.exists("LifeOps.zip"):
    !unzip -o LifeOps.zip
    # Check if unzipped into a subfolder
    if os.path.exists("LifeOps") and os.path.isdir("LifeOps"):
        %cd LifeOps

# 2. API URL — same pattern as a hosted-injection GRPO notebook: public Space, or local.
#    Priority: `LIFEOPS_ENV_URL` (e.g. Colab secret) > `LIFEOPS_SPACE_URL` below > local `server.app`.
LIFEOPS_SPACE_URL = ""  # e.g. "https://yourname-lifeops.hf.space"  (no trailing slash)

_from_env = (os.environ.get("LIFEOPS_ENV_URL") or "").strip()
_from_cell = (LIFEOPS_SPACE_URL or "").strip()
ENV_URL = (_from_env or _from_cell or "http://127.0.0.1:7860").rstrip("/")
os.environ["LIFEOPS_ENV_URL"] = ENV_URL

_remote = "hf.space" in ENV_URL.lower() or "huggingface.co" in ENV_URL.lower()

if not _remote:
    print("Launching local LifeOps server ->", ENV_URL)
    with open("server_log.txt", "w") as log:
        subprocess.Popen(
            [sys.executable, "-m", "server.app"],
            stdout=log,
            stderr=log,
            env=dict(os.environ, PYTHONPATH=os.getcwd(), PORT="7860"),
        )
else:
    print("Using remote LifeOps API:", ENV_URL)

# 3. Wait for /health (HF Space cold start can be slow)
_attempts = 20 if _remote else 10
_sleep = 3
_httpx_timeout = 25.0 if _remote else 5.0
connected = False
for i in range(_attempts):
    time.sleep(_sleep)
    try:
        with httpx.Client(base_url=ENV_URL, timeout=_httpx_timeout) as http:
            health = http.get("/health")
            if health.status_code == 200:
                print(f"Connected to LifeOps environment ✓ ({ENV_URL})")
                connected = True
                break
    except Exception as _e:
        print(f"Attempt {i+1}/{_attempts}: waiting... ({type(_e).__name__})")

if not connected:
    print("❌ Server failed. LOG OUTPUT:")
    if os.path.exists("server_log.txt"):
        with open("server_log.txt", "r") as f: print(f.read())

## 4. Define Reward Function

### Why logged rewards often go flat (LifeOps + GRPO)

TRL’s GRPO builds **within-prompt advantages** from the spread of rewards across the `num_generations` completions. When that spread collapses to **near-zero standard deviation**, the group-relative advantage is ~0 for every token, so the policy update (and the logged `reward` mean) barely moves even though training “runs” for many steps.

In LifeOps, the **environment step reward is largely a deterministic function of `(parsed_action, justification)`** for a fixed scenario: if the model converges to the **same valid action line** (and similar justification) across most samples in a batch, every completion gets almost the same env score. The format head can also plateau once outputs are consistently two-line. Together, that produces the **flat horizontal curves** you see — it is usually **mode collapse / low within-group variance**, not a bug in matplotlib.

**Mitigations** (conceptual): more `num_generations`, higher sampling temperature / `top_p`, slightly longer completions, richer multi-step scenarios, or auxiliary rewards that vary with phrasing. This notebook also maps env and format signals into **[0, 1]** so scales match a grader-style setup like the injection reference notebook.

In [ ]:
from scripts.rl_action_utils import (
    resolve_allowed_actions,
    strip_generative_spill,
    extract_raw_action_phrase,
    map_phrase_to_allowed_action,
    action_line_is_snake_enum,
    extract_justification_phrase,
    compute_format_reward,
    normalize_lifeops_env_reward,
    normalize_format_reward_unit,
)

# Debug window for inspecting raw completions during training.
DEBUG_START_STEP = 3
DEBUG_END_STEP = 8
_env_reward_call_step = 0

# Normalized weights so TRL's combined `reward` is in [0, 1] when both heads are in [0, 1].
_RW_ENV, _RW_FMT = 1.0, 0.45
_LIFEOPS_WSUM = _RW_ENV + _RW_FMT
LIFEOPS_REWARD_WEIGHTS = (_RW_ENV / _LIFEOPS_WSUM, _RW_FMT / _LIFEOPS_WSUM)


def env_reward_fn(
    completions: list[str],
    prompts: list[str] | None = None,
    allowed_actions_json=None,
    **kwargs,
) -> list[float]:
    global _env_reward_call_step
    _env_reward_call_step += 1

    if prompts is None:
        prompts = kwargs.get("prompts")
    if allowed_actions_json is None:
        allowed_actions_json = kwargs.get("allowed_actions_json")

    tok = kwargs.get("processing_class") or kwargs.get("tokenizer")

    rewards: list[float] = []
    with httpx.Client(base_url=ENV_URL, timeout=10) as http:
        for idx, completion in enumerate(completions):
            prompt = None
            if prompts is not None and idx < len(prompts):
                prompt = prompts[idx]

            allowed_blob = None
            if allowed_actions_json is not None and idx < len(allowed_actions_json):
                allowed_blob = allowed_actions_json[idx]

            allowed = resolve_allowed_actions(prompt=prompt, allowed_actions_json=allowed_blob)
            cleaned = strip_generative_spill(completion, tokenizer=tok)
            phrase = extract_raw_action_phrase(cleaned)
            mapped, map_pen = map_phrase_to_allowed_action(phrase or "", allowed)

            if not mapped:
                rewards.append(normalize_lifeops_env_reward(-2.0))
                continue

            justification = extract_justification_phrase(cleaned, tokenizer=tok) or " "

            try:
                http.post("/reset")
                resp = http.post(
                    "/step",
                    json={"action": {"choice": mapped, "justification": justification}},
                )
                raw = float(resp.json().get("reward", 0.0)) if resp.status_code == 200 else -2.0
                rewards.append(normalize_lifeops_env_reward(raw + float(map_pen)))
            except Exception:
                rewards.append(normalize_lifeops_env_reward(-2.0))

    if DEBUG_START_STEP <= _env_reward_call_step <= DEBUG_END_STEP:
        print(f"\n===== DEBUG STEP {_env_reward_call_step} (env) =====")
        for i, completion in enumerate(completions[:5]):
            p = prompts[i] if prompts is not None and i < len(prompts) else None
            blob = (
                allowed_actions_json[i]
                if allowed_actions_json is not None and i < len(allowed_actions_json)
                else None
            )
            allowed = resolve_allowed_actions(prompt=p, allowed_actions_json=blob)
            cleaned = strip_generative_spill(completion, tokenizer=tok)
            phrase = extract_raw_action_phrase(cleaned)
            mapped, map_pen = map_phrase_to_allowed_action(phrase or "", allowed)
            snippet = cleaned.replace("\n", "\\n")[:240]
            ok = mapped is not None and (allowed is None or mapped in allowed)
            print(
                f"sample={i} raw_action={phrase!r} mapped={mapped} map_pen={map_pen:+.2f} "
                f"allowed_ok={ok} env_u={rewards[i]:.4f}  (0–1 normalized)"
            )
            print(f"text: {snippet}")

    return rewards


def format_reward_fn(completions: list[str], **kwargs) -> list[float]:
    tok = kwargs.get("processing_class") or kwargs.get("tokenizer")
    return [
        normalize_format_reward_unit(compute_format_reward(c, tokenizer=tok))
        for c in completions
    ]

## 5. Prepare Dataset

In [ ]:
from datasets import Dataset
import sys
import inspect
import copy

sys.path.append(os.getcwd())
from scripts.dataset_builder import LifeopsDatasetBuilder


def _truncate_messages(messages: list[dict], max_user_chars: int = 900) -> list[dict]:
    """Bound user message size for Colab stability."""
    msgs = copy.deepcopy(messages)
    for m in msgs:
        if m.get("role") == "user" and isinstance(m.get("content"), str):
            c = m["content"]
            if len(c) > max_user_chars:
                m["content"] = c[: max_user_chars - 30] + "\n\n[TRUNCATED]"
    return msgs


raw_items = LifeopsDatasetBuilder.generate_rl_dataset(100)
rows = []
for item in raw_items:
    rows.append(
        {
            "prompt": _truncate_messages(item["prompt"]),
            "scenario_id": item.get("scenario_id"),
            "allowed_actions_json": item.get("allowed_actions_json"),
        }
    )

dataset = Dataset.from_list(rows)
print(f"✅ Generated {len(dataset)} examples")

try:
    from trl import GRPOConfig

    sig = inspect.signature(GRPOConfig.__init__)
    print("TRL GRPOConfig supports max_prompt_length:", "max_prompt_length" in sig.parameters)
    print("TRL GRPOConfig supports max_completion_length:", "max_completion_length" in sig.parameters)
    print("TRL GRPOConfig supports mask_truncated_completions:", "mask_truncated_completions" in sig.parameters)
    print("TRL GRPOConfig supports reward_weights:", "reward_weights" in sig.parameters)
    print("TRL GRPOConfig supports remove_unused_columns:", "remove_unused_columns" in sig.parameters)
except Exception as e:
    print("Could not introspect GRPOConfig:", repr(e))

## 5b. Uniform-action baseline

Estimates a **reference score** by averaging the weighted reward over every **legal** `Action:` line for a fixed template justification on a random subset of dataset rows. This is the horizontal line on the next plot and the `baseline_score` field in `grpo_out/training_results.json` (same role as a fixed baseline in a reference GRPO notebook).

In [ ]:
import os
import sys

sys.path.append(os.getcwd())
from scripts.lifeops_grpo_metrics import uniform_action_baseline_stats

BASELINE_RESULT = uniform_action_baseline_stats(
    dataset,
    env_reward_fn,
    format_reward_fn,
    n_rows=40,
    seed=0,
    reward_weights=LIFEOPS_REWARD_WEIGHTS,
)
print("baseline_mean (0–1 weighted env+format):", round(BASELINE_RESULT["baseline_mean"], 4))
print("rows_used:", BASELINE_RESULT["rows_used"])

## 6. GRPO Training

In [ ]:
import inspect
from trl import GRPOConfig, GRPOTrainer

# Newer Unsloth versions already patch GRPO internals automatically.
# On Colab / Python 3.12, explicit PatchFastRL() can raise OSError.
try:
    from unsloth import PatchFastRL
    try:
        PatchFastRL()
    except OSError:
        pass
except Exception:
    pass


def _build_grpo_config(**kwargs) -> GRPOConfig:
    """Build GRPOConfig across TRL versions (some kwargs vary by TRL release)."""
    sig = inspect.signature(GRPOConfig.__init__)
    filtered = {k: v for k, v in kwargs.items() if k in sig.parameters}
    dropped = sorted(set(kwargs) - set(filtered))
    if dropped:
        print("Note: dropping unsupported GRPOConfig args:", dropped)
    return GRPOConfig(**filtered)


cfg = _build_grpo_config(
    # More samples per prompt → more within-group spread for GRPO (avoids std=0 on env).
    num_generations=8,
    max_steps=100,
    learning_rate=2e-5,
    max_prompt_length=384,  # only used on older TRL; ignored otherwise
    max_completion_length=72,
    # Slightly higher diversity so not every completion is identical 1-liner.
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    mask_truncated_completions=True,
    # Same normalized pair as section 4 — combined TRL `reward` stays in [0, 1].
    reward_weights=list(LIFEOPS_REWARD_WEIGHTS),
    remove_unused_columns=False,  # keep dataset extras for reward_funcs (allowed_actions_json, ...)
    output_dir="./grpo_out",
    logging_steps=1,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[env_reward_fn, format_reward_fn],
    args=cfg,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("GRPO config loaded:")
if hasattr(trainer.args, "max_prompt_length"):
    print("- max_prompt_length:", trainer.args.max_prompt_length)
else:
    print("- max_prompt_length: (not supported by this TRL version; prompts bounded in dataset cell)")
print("- max_completion_length:", getattr(trainer.args, "max_completion_length", None))
print("- num_generations:", trainer.args.num_generations)
print("- temperature:", getattr(trainer.args, "temperature", None))
print("- top_p:", getattr(trainer.args, "top_p", None))
print("- repetition_penalty:", getattr(trainer.args, "repetition_penalty", None))
print("- mask_truncated_completions:", getattr(trainer.args, "mask_truncated_completions", None))
print("- reward_weights:", getattr(trainer.args, "reward_weights", None))
print("- remove_unused_columns:", getattr(trainer.args, "remove_unused_columns", None))

gen_cfg = getattr(trainer.model, "generation_config", None)
if gen_cfg is not None:
    print("Model generation_config:")
    print("- max_new_tokens:", getattr(gen_cfg, "max_new_tokens", None))
    print("- eos_token_id:", getattr(gen_cfg, "eos_token_id", None))

trainer.train()

## 7. Plot training reward curve (vs baseline)

After `trainer.train()`, reads `trainer.state.log_history` and plots **any logged scalar fields whose name contains `reward`**. A **dashed** line shows the **uniform-action baseline** from section 5b (same idea as comparing to a fixed baseline in a reference notebook). Figure is saved under `grpo_out/`.

In [ ]:
import os

import matplotlib.pyplot as plt
from collections import defaultdict

os.makedirs("grpo_out", exist_ok=True)

LIFEOPS_REWARD_STEPS, LIFEOPS_REWARD_VALS = [], []

# Requires: `trainer.train()` has finished and `trainer` is in scope.
hist = getattr(trainer.state, "log_history", None) or []
if not hist:
    print("log_history is empty — run training first.")
else:
    # metric_name -> steps[], vals[]
    series = defaultdict(lambda: {"steps": [], "vals": []})
    for entry in hist:
        st = entry.get("step")
        if st is None:
            continue
        for k, v in entry.items():
            if k == "step":
                continue
            if not isinstance(v, (int, float)):
                continue
            low = k.lower()
            if "reward" in low and "std" not in low:
                series[k]["steps"].append(int(st) if st == int(st) else st)
                series[k]["vals"].append(float(v))

    # Debug: show keys on last log row (helps if TRL renames metrics)
    last = hist[-1]
    print("Last log entry keys:", sorted(last.keys()))

    if not any(s["steps"] for s in series.values()):
        print("No scalar *reward* metrics found in log_history.")
    else:
        plt.figure(figsize=(10, 5))
        # Prefer a combined `reward` line when present, then plot the rest
        prefer = ["reward", "train/reward", "train_reward"]
        plotted = set()
        for name in prefer:
            if name in series and series[name]["steps"]:
                plt.plot(
                    series[name]["steps"],
                    series[name]["vals"],
                    marker="o",
                    ms=3,
                    linewidth=2,
                    label=name,
                )
                plotted.add(name)

        for name, data in sorted(series.items()):
            if name in plotted or not data["steps"]:
                continue
            plt.plot(
                data["steps"],
                data["vals"],
                marker="o",
                ms=2,
                linewidth=1.5,
                label=name,
            )

        if "BASELINE_RESULT" in globals() and BASELINE_RESULT.get("rows_used", 0):
            bm = float(BASELINE_RESULT["baseline_mean"])
            plt.axhline(
                bm,
                color="gray",
                linestyle="--",
                linewidth=2,
                label=f"baseline (uniform actions)={bm:.3f}",
            )

        plt.xlabel("Training step")
        plt.ylabel("Reward (0–1 heads, from trainer logs)")
        plt.title("LifeOps GRPO — training reward(s) vs baseline")
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=8, loc="best")
        plt.tight_layout()
        out = "grpo_out/lifeops_reward_curve_vs_baseline.png"
        plt.savefig(out, dpi=150)
        plt.show()
        print(f"Saved {out}")

        # For JSON export (section 9)
        primary = None
        for name in ("reward", "train/reward", "train_reward"):
            if name in series and series[name]["steps"]:
                primary = name
                break
        if primary:
            LIFEOPS_REWARD_STEPS = list(series[primary]["steps"])
            LIFEOPS_REWARD_VALS = list(series[primary]["vals"])
        else:
            LIFEOPS_REWARD_STEPS, LIFEOPS_REWARD_VALS = [], []


## 8. Post-train sanity check (LifeOps-specific)

This is intentionally small: it measures **parse validity**, **chat spill contamination**, and a quick **HTTP env reward** on a handful of dataset rows — not a generic benchmark dump.

In [ ]:
import random
import re

import torch
from unsloth import FastLanguageModel

# Uses globals from earlier cells: model, tokenizer, dataset, ENV_URL, env_reward_fn

N = min(8, len(dataset)) if "dataset" in globals() else 0
if N <= 0:
    print("No dataset in memory — run the dataset cell first.")
else:
    rng = random.Random(42)
    idxs = rng.sample(range(len(dataset)), k=N)

    FastLanguageModel.for_inference(model)

    rows = []
    for i in idxs:
        ex = dataset[i]
        messages = ex["prompt"]
        blob = ex.get("allowed_actions_json")
        scenario_id = ex.get("scenario_id")

        if hasattr(tokenizer, "apply_chat_template"):
            prompt_txt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt_txt = "\n".join(
                f"{m.get('role','user').strip()}:\n{m.get('content','')}" for m in messages
            ).strip() + "\nassistant:\n"

        inputs = tokenizer([prompt_txt], return_tensors="pt").to(model.device)
        try:
            max_new_tokens = int(getattr(trainer.args, "max_completion_length", 72))
        except NameError:
            max_new_tokens = 72

        with torch.inference_mode():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )

        gen = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        rows.append((scenario_id, messages, blob, gen))

    completions = [g for (_, _, _, g) in rows]
    prompts = [m for (_, m, _, _) in rows]
    blobs = [b for (_, _, b, _) in rows]

    env_rs = env_reward_fn(completions=completions, prompts=prompts, allowed_actions_json=blobs)
    fmt_rs = [format_reward_fn([c])[0] for c in completions]

    from collections import defaultdict

    W_ENV, W_FMT = LIFEOPS_REWARD_WEIGHTS
    combined = [W_ENV * e + W_FMT * f for e, f in zip(env_rs, fmt_rs)]
    TRAINED_OVERALL = sum(combined) / len(combined)
    scenario_ids = [dataset[i].get("scenario_id") for i in idxs]
    by_sid = defaultdict(list)
    for sid, c in zip(scenario_ids, combined):
        by_sid[str(sid if sid is not None else "unknown")].append(c)
    TRAINED_TASK_SCORES = {k: round(sum(v) / len(v), 4) for k, v in by_sid.items()}

    spill_hits = sum(1 for c in completions if re.search(r"(?im)\b(human|assistant|user|system)\s*:", c))
    two_line_hits = sum(1 for c in completions if len([ln for ln in c.splitlines() if ln.strip()]) == 2)

    print(f"samples={N}")
    print(
        f"mean_env_u(0-1)={sum(env_rs)/len(env_rs):.3f} mean_format_u(0-1)={sum(fmt_rs)/len(fmt_rs):.3f}"
    )
    print(f"weighted_mean(0-1)={TRAINED_OVERALL:.4f}")
    if "BASELINE_RESULT" in globals():
        print(f"baseline_mean(0-1)={BASELINE_RESULT['baseline_mean']:.4f}")
    print(f"two_line_rate={two_line_hits/N:.2f} role_spill_rate={spill_hits/N:.2f}")
    print("per_scenario_means:", TRAINED_TASK_SCORES)

    for j, (sid, messages, blob, gen) in enumerate(rows[:3]):
        print("\n--- sample", j, "---")
        print("scenario_id:", sid)
        print("allowed_actions_json:", (blob or "")[:160])
        print("completion:\n", gen)
        print("env_r=", env_rs[j], "fmt_r=", fmt_rs[j])


## 9. Save results (`training_results.json`)

Writes **`grpo_out/training_results.json`** in the same spirit as a reference GRPO notebook: baseline score, reward curve from `log_history`, and post-train evaluation aggregates. Run after sections 7–8.

In [ ]:
import json
from pathlib import Path

hist = getattr(trainer.state, "log_history", None) or []
steps_json = [l["step"] for l in hist if "reward" in l]
rewards_json = [l["reward"] for l in hist if "reward" in l]

results = {
    "model": "unsloth/Qwen2.5-3B-Instruct + GRPO LoRA (LifeOps)",
    "training_steps": getattr(trainer.args, "max_steps", None),
    "num_generations": getattr(trainer.args, "num_generations", None),
    "reward_weights": list(LIFEOPS_REWARD_WEIGHTS),
    "baseline_score": (
        round(float(BASELINE_RESULT["baseline_mean"]), 4)
        if "BASELINE_RESULT" in globals() and BASELINE_RESULT.get("rows_used", 0)
        else None
    ),
    "trained_agent_task_scores": (
        TRAINED_TASK_SCORES if "TRAINED_TASK_SCORES" in globals() else {}
    ),
    "trained_agent_overall": (
        round(float(TRAINED_OVERALL), 4) if "TRAINED_OVERALL" in globals() else None
    ),
    "reward_curve": {
        "steps": LIFEOPS_REWARD_STEPS if LIFEOPS_REWARD_STEPS else steps_json,
        "rewards": LIFEOPS_REWARD_VALS if LIFEOPS_REWARD_VALS else rewards_json,
    },
    "reward_curve_png": "grpo_out/lifeops_reward_curve_vs_baseline.png",
}

Path("grpo_out").mkdir(parents=True, exist_ok=True)
out_path = Path("grpo_out/training_results.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("Saved", out_path)
print(json.dumps(results, indent=2)[:2500])